In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行


In [2]:
import pandas as pd
import numpy as np
import sqlite3
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import json
import copy
import warnings
import random
import time
from datetime import datetime
warnings.filterwarnings('ignore')
from scipy.stats import t      # 🌟 换成 t 分布
from scipy.optimize import brentq
from tqdm import tqdm
from scipy.stats import norm

pd.set_option('display.max_columns', None)  # 显示所有列
pd.set_option('display.max_rows', 100)     # 显示前100行

timestamp = '20260803_0631'

fused_k1_forecasts = pd.read_parquet('/Users/jianbinchen/Downloads/results/fused_k1_forecasts_' + timestamp + '.parquet') 
fused_k3_forecasts = pd.read_parquet('/Users/jianbinchen/Downloads/results/fused_k3_forecasts_' + timestamp + '.parquet')
structured_forecasts = pd.read_parquet('/Users/jianbinchen/Downloads/results/structured_forecasts_' + timestamp + '.parquet')
path = '/Users/jianbinchen/NonSync/GitHub/Research/PricingTailRisk_MDN/output'

display(pd.concat([fused_k1_forecasts.head(2), fused_k3_forecasts.tail(2)], ignore_index=True))
display(pd.concat([structured_forecasts.head(2), structured_forecasts.tail(2)], ignore_index=True))
display(pd.concat([fused_k1_forecasts.head(2), structured_forecasts.head(2)], ignore_index=True))

,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[0.025033660233020782],[0.1106194332242012],None
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[0.036503858864307404],[0.08119947463274002],None
2,93434,2024-12-31,0.147685,fused_normal_k3,"[0.053917981684207916, 0.19164061546325684, 0....","[-1.1785269975662231, 0.0896308422088623, 1.45...","[0.18137973546981812, -0.011492399498820305, -...","[0.6383622884750366, 0.047841329127550125, 0.1...",None
3,93436,2024-12-31,0.001882,fused_normal_k3,"[0.05155758187174797, 0.24249064922332764, 0.7...","[-1.3262883424758911, 0.22197556495666504, 1.2...","[0.1538901925086975, 0.00034094974398612976, 0...","[0.35251352190971375, 0.06686557084321976, 0.1...",None


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,structured_normal_k3,"[0.8789113163948059, 0.06002949923276901, 0.06...","[1.5979101657867432, -1.0859376192092896, -1.0...","[0.015402581542730331, 0.11905046552419662, -0...","[0.07071688026189804, 0.24829387664794922, 0.0...",None
1,10002,2007-01-31,0.033054,structured_normal_k3,"[0.8789113163948059, 0.06002949923276901, 0.06...","[1.5979101657867432, -1.0859376192092896, -1.0...","[0.024411216378211975, 0.11881102621555328, -0...","[0.0681501254439354, 0.16973699629306793, 0.08...",None
2,93434,2024-12-31,0.147685,structured_normal_k3,"[0.016913507133722305, 0.065256267786026, 0.91...","[-1.9223929643630981, -0.5721834301948547, 2.0...","[-0.1619882881641388, 0.1430777758359909, 0.00...","[0.15946613252162933, 0.48921945691108704, 0.1...",None
3,93436,2024-12-31,0.001882,structured_normal_k3,"[0.016913507133722305, 0.065256267786026, 0.91...","[-1.9223929643630981, -0.5721834301948547, 2.0...","[-0.12319877743721008, 0.14003165066242218, 0....","[0.10389988869428635, 0.15982693433761597, 0.0...",None


,permno,mthcaldt,target_ret_final,model_id,pi_vec,pi_logits_vec,mu_vec,sigma_vec,nu_vec
0,10001,2007-01-31,0.264774,fused_normal_k1,[1.0],[0.0],[0.025033660233020782],[0.1106194332242012],None
1,10002,2007-01-31,0.033054,fused_normal_k1,[1.0],[0.0],[0.036503858864307404],[0.08119947463274002],None
2,10001,2007-01-31,0.264774,structured_normal_k3,"[0.8789113163948059, 0.06002949923276901, 0.06...","[1.5979101657867432, -1.0859376192092896, -1.0...","[0.015402581542730331, 0.11905046552419662, -0...","[0.07071688026189804, 0.24829387664794922, 0.0...",None
3,10002,2007-01-31,0.033054,structured_normal_k3,"[0.8789113163948059, 0.06002949923276901, 0.06...","[1.5979101657867432, -1.0859376192092896, -1.0...","[0.024411216378211975, 0.11881102621555328, -0...","[0.0681501254439354, 0.16973699629306793, 0.08...",None


In [3]:
import json
import numpy as np
import pandas as pd
from scipy.stats import norm, t


def calculate_es_table(forecasts, alphas=(0.01, 0.05, 0.10),
                       chunk_size=50_000, max_rows=None, iterations=45):

    """
    根据一个模型的 MDN forecast DataFrame 计算左尾 VaR 和 loss-ES。

    var_alpha = F^{-1}(alpha)，是收益率分位数；
    es_alpha  = -E[R | R <= var_alpha]，是带符号的 loss-ES。

    每次只能传入一个 model_id，但可以是 K=1、K=3、Normal 或 Student-t。
    """
    frame = forecasts.iloc[:max_rows] if max_rows is not None else forecasts
    alphas = np.asarray(alphas, dtype=float)

    def decode(column):
        values = frame[column].map(lambda x: json.loads(x) if isinstance(x, str) else x)
        return np.stack(values).astype(float, copy=False)

    pi = decode("pi_vec")
    mu = decode("mu_vec")
    sigma = decode("sigma_vec")

    if pi.shape != mu.shape or mu.shape != sigma.shape:
        raise ValueError("pi、mu 和 sigma 的维度不一致。")
    if not np.isfinite(pi).all() or not np.isfinite(mu).all() or not np.isfinite(sigma).all():
        raise ValueError("pi、mu 或 sigma 含有 NaN/inf。")
    if np.any(sigma <= 0):
        raise ValueError("sigma 必须大于0。")
    if not np.allclose(pi.sum(axis=1), 1.0, atol=1e-5):
        raise ValueError("部分 mixture probabilities 加总不等于1。")

    # Normal 模型的 nu_vec 全为空；Student-t 模型的 nu_vec 应全部非空
    nu = None
    if "nu_vec" in frame.columns:
        has_nu = frame["nu_vec"].notna()

        if has_nu.any() and not has_nu.all():
            raise ValueError("nu_vec 不能同时包含空值和非空值。")

        if has_nu.all():
            nu = decode("nu_vec")
            if nu.shape != mu.shape or not np.isfinite(nu).all() or np.any(nu <= 2):
                raise ValueError("nu 必须与 mu 维度一致，而且全部大于2。")
            
    n_rows, n_components = pi.shape
    n_alphas = len(alphas)

    var_matrix = np.empty((n_rows, n_alphas), dtype=float)
    es_matrix = np.empty((n_rows, n_alphas), dtype=float)
    bracket_probability = max(float(alphas.min()) * 0.1, 1e-10)

    for start in range(0, n_rows, chunk_size):
        stop = min(start + chunk_size, n_rows)

        p = pi[start:stop]
        m = mu[start:stop]
        s = sigma[start:stop]
        degrees = None if nu is None else nu[start:stop]

        # Normal K=1 有闭式解，不需要二分法
        if degrees is None and n_components == 1:
            z = norm.ppf(alphas)[None, :]
            chunk_var = m + s * z
            chunk_es = -m + s * norm.pdf(z) / alphas[None, :]

        else:
            # 为每一行建立足够宽的分位数搜索区间
            if degrees is None:
                lower_component = m + s * norm.ppf(bracket_probability)
                upper_component = m + s * norm.ppf(1.0 - bracket_probability)
            else:
                lower_component = m + s * t.ppf(bracket_probability, df=degrees)
                upper_component = m + s * t.ppf(1.0 - bracket_probability, df=degrees)

            lower = np.repeat(lower_component.min(axis=1)[:, None], n_alphas, axis=1)
            upper = np.repeat(upper_component.max(axis=1)[:, None], n_alphas, axis=1)

            # 批量二分法求 mixture quantiles
            for _ in range(iterations):
                middle = (lower + upper) / 2.0
                z = (middle[..., None] - m[:, None, :]) / s[:, None, :]

                if degrees is None:
                    component_cdf = norm.cdf(z)
                else:
                    component_cdf = t.cdf(z, df=degrees[:, None, :])

                mixture_cdf = np.sum(p[:, None, :] * component_cdf, axis=2)
                lower = np.where(mixture_cdf < alphas, middle, lower)
                upper = np.where(mixture_cdf >= alphas, middle, upper)

            chunk_var = (lower + upper) / 2.0
            z = (chunk_var[..., None] - m[:, None, :]) / s[:, None, :]

            if degrees is None:
                component_cdf = norm.cdf(z)
                adjusted_pdf = norm.pdf(z)
            else:
                degrees_3d = degrees[:, None, :]
                component_cdf = t.cdf(z, df=degrees_3d)
                adjusted_pdf = (
                    (degrees_3d + z**2) / (degrees_3d - 1.0)
                    * t.pdf(z, df=degrees_3d)
                )

            left_moment = (
                m[:, None, :] * component_cdf
                - s[:, None, :] * adjusted_pdf
            )

            chunk_es = (
                -np.sum(p[:, None, :] * left_moment, axis=2)
                / alphas[None, :]
            )

        var_matrix[start:stop] = chunk_var
        es_matrix[start:stop] = chunk_es

    # Mixture conditional mean and standard deviation
    mean_pred = np.sum(pi * mu, axis=1)

    if nu is None:
        component_variance = sigma**2
    else:
        component_variance = sigma**2 * nu / (nu - 2.0)

    mixture_variance = (
        np.sum(pi * (mu**2 + component_variance), axis=1)
        - mean_pred**2
    )
    vol_pred = np.sqrt(np.maximum(mixture_variance, 0.0))

    result = frame[["permno", "mthcaldt", "model_id"]].reset_index(drop=True)

    for column, alpha in enumerate(alphas):
        suffix = f"{alpha * 100:g}".replace(".", "_")
        result[f"es_{suffix}"] = es_matrix[:, column]
        result[f"var_{suffix}"] = var_matrix[:, column]

    result["mean_pred"] = mean_pred
    result["vol_pred"] = vol_pred
    result["realized_ret"] = frame["target_ret_final"].to_numpy(dtype=float)

    return result

In [4]:
run_test = "full"  # "quick" 或 "full"
max_rows = 100 if run_test == "quick" else None
alphas = (0.01, 0.05, 0.10)

fused_k1_es = calculate_es_table(
    fused_k1_forecasts, alphas=alphas, max_rows=max_rows
)
fused_k1_es.to_parquet(
    os.path.join(path, f"ES_fused_normal_k1_{timestamp}.parquet"), index=False
)

fused_k3_es = calculate_es_table(
    fused_k3_forecasts, alphas=alphas, max_rows=max_rows
)
fused_k3_es.to_parquet(
    os.path.join(path, f"ES_fused_normal_k3_{timestamp}.parquet"), index=False
)

structured_es = calculate_es_table(
    structured_forecasts, alphas=alphas, max_rows=max_rows
)
structured_es.to_parquet(
    os.path.join(path, f"ES_structured_normal_k3_{timestamp}.parquet"), index=False
)

In [5]:
fused_k1_es.head(2)

,permno,mthcaldt,model_id,es_1,var_1,es_5,var_5,es_10,var_10,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,fused_normal_k1,0.269791,-0.232306,0.203142,-0.156919,0.169102,-0.116731,0.025034,0.110619,0.264774
1,10002,2007-01-31,fused_normal_k1,0.179910,-0.152394,0.130987,-0.097057,0.106000,-0.067557,0.036504,0.081199,0.033054


In [6]:
display(pd.concat([fused_k1_es.head(2), fused_k3_es.tail(2)], ignore_index=True))
display(pd.concat([structured_es.head(2), structured_es.tail(2)], ignore_index=True))
display(pd.concat([fused_k1_es.head(2), structured_es.head(2)], ignore_index=True))

,permno,mthcaldt,model_id,es_1,var_1,es_5,var_5,es_10,var_10,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,fused_normal_k1,0.269791,-0.232306,0.203142,-0.156919,0.169102,-0.116731,0.025034,0.110619,0.264774
1,10002,2007-01-31,fused_normal_k1,0.179910,-0.152394,0.130987,-0.097057,0.106000,-0.067557,0.036504,0.081199,0.033054
2,93434,2024-12-31,fused_normal_k3,0.775177,-0.525980,0.469003,-0.321307,0.371899,-0.236532,0.007387,0.230638,0.147685
3,93436,2024-12-31,fused_normal_k3,0.409393,-0.305162,0.267120,-0.188361,0.213225,-0.135828,0.022677,0.144863,0.001882


,permno,mthcaldt,model_id,es_1,var_1,es_5,var_5,es_10,var_10,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,structured_normal_k3,0.275130,-0.188236,0.166516,-0.113265,0.131521,-0.082883,0.019208,0.096330,0.264774
1,10002,2007-01-31,structured_normal_k3,0.209857,-0.166341,0.142479,-0.101603,0.113930,-0.072098,0.025769,0.084265,0.033054
2,93434,2024-12-31,structured_normal_k3,0.627250,-0.418886,0.366854,-0.245599,0.289432,-0.184918,0.011322,0.187064,0.147685
3,93436,2024-12-31,structured_normal_k3,0.256549,-0.216078,0.189068,-0.144812,0.157138,-0.108902,0.017809,0.104562,0.001882


,permno,mthcaldt,model_id,es_1,var_1,es_5,var_5,es_10,var_10,mean_pred,vol_pred,realized_ret
0,10001,2007-01-31,fused_normal_k1,0.269791,-0.232306,0.203142,-0.156919,0.169102,-0.116731,0.025034,0.110619,0.264774
1,10002,2007-01-31,fused_normal_k1,0.179910,-0.152394,0.130987,-0.097057,0.106000,-0.067557,0.036504,0.081199,0.033054
2,10001,2007-01-31,structured_normal_k3,0.275130,-0.188236,0.166516,-0.113265,0.131521,-0.082883,0.019208,0.096330,0.264774
3,10002,2007-01-31,structured_normal_k3,0.209857,-0.166341,0.142479,-0.101603,0.113930,-0.072098,0.025769,0.084265,0.033054
